In [ ]:
import os
import json
from urllib.parse import unquote
from DocMiner import DocMiner, Task
from DocMinerPreprocess import DocMinerPreprocess
# from azure.ai.documentintelligence.models import AnalyzeResult 
from openai import AzureOpenAI

sync_openai = AzureOpenAI(
    api_version=os.getenv("AZURE_OPENAI_API_VERSION"),
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    azure_endpoint=os.getenv("AZURE_OPENAI_API_URL"),
    max_retries=3,
    timeout=30
)

In [ ]:
def create_docminer(docx_path: str, openai_client: AzureOpenAI):
    from DocMinerPreprocess import DocMinerPreprocess

    system_message = """
    You are assisting content manager. The content manger is organizing document for better search and document operation tasks. Document operation task can be updating a term or ID in the document. 
    Current task is document preprocessing for efficient search and document operation. 

    Do data mining by extracting keyword and IDs from the given document. Define the hierarchy. Use the document file name, and under the file name build a keyword, ID hierarchy that might have dependency on other documents or IDs.

    # Data mining steps 
    1. note document file name
    2. Categorize the document
    3. Extract headings, titles, and subtitle to know the overall structure, the number or the order of title matters
    4. extract keyword
    5. extract IDs. It is very important to capture task_code and its description

    # Output Format

    Return your finding to user as a JSON format.
    {
            "document_filename": "Planning Model Training Using Computer Vision v0.docx",
            "file_location": "https://docminer.blob.core.windows.net/docminer/Planning Model Training Using Computer Vision v0.docx",
            "project_title": "Planning Model Training Using Computer Vision",
            "author": "Hyunsuk",
            "table_of_contents": [
                "Planning Model Training Using Computer Vision",
                "Phases of the Project and Training Environment Specifications",
                "Introduction",
                "Phases of the Project",
                "Phases of the Project > Model Selection and Development",
                "Phases of the Project > Model Training and Validation",
                "Phases of the Project > Model Testing and Evaluation",
                "Training Environment and Specifications",
                "Training Environment and Specifications > Hardware Specifications",
                "Training Environment and Specifications > Software Specifications",
                "Training Environment and Specifications > Cloud Services",
                "Training Environment and Specifications > Data Management",
                "Conclusion"
            ],
            "keywords": [
                "computer vision",
                "model training",
                "model selection",
                "model development",
                "training environment",
                "hardware specifications",
                "software specifications",
                "cloud services",
                "data management"
            ],
            "tasks": [
                {
                    "task_code": "MT0",
                    "task_description": "Data Scientist, utilizing machine learning algorithms to train the model on the labeled data."
                }
            ]
        }
    """

    analysis_result, blob_path = DocMinerPreprocess().preprocess_document(docx_path)
    user_message = f"""
    {str(analysis_result['content'])}
    file_location: {blob_path}
    """

    res = openai_client.chat.completions.create(
        model = os.getenv("DEPLOYMENT_NAME"),
        messages = [
            {"role": "system", "content": system_message},
            {"role": "user", "content": user_message}
            ],
        max_tokens=2000,
        temperature=0.01,
        top_p=0.01,
    )

    try:
        json_content = res.choices[0].message.content.split("```json")[1].strip().strip('```')
    except IndexError:
        print("Index error in parsing JSON content, trying to parse the content without split function")
        json_content = res.choices[0].message.content
    except:
        print("Error in parsing JSON content")
    
    doc_data = json.loads(json_content)
    return DocMiner(**doc_data)

In [ ]:
list_docminer = []

# get list of files
doc_dir = "../linked_docs_v0"
for file in os.listdir(doc_dir):
    if file.endswith(".docx"):
        print(file)
        list_docminer.append(create_docminer(os.path.join(doc_dir, file), sync_openai))

list_docminer

Plan for Model Training Using Azure Machine Learning Service_V0.docx
Planning Model Training Using Computer Vision v0.docx
Project Data Science Project for Remote Desktop Surveillance_v0.docx


[DocMiner(document_id='45b26899-ab7b-43cd-b04d-717713d38333', document_filename='Plan for Model Training Using Azure Machine Learning Service_V0.docx', file_location='https://openaiembedding.blob.core.windows.net/docminer-container/Plan for Model Training Using Azure Machine Learning Service_V0.docx', project_title='Plan for Model Training Using Azure Machine Learning Service', author='Unknown', table_of_contents=['Plan for Model Training Using Azure Machine Learning Service', 'Roles: Data Scientist and ML Engineer', 'Introduction', 'Project Overview', 'Phases of the Project', 'Phases of the Project > Phase 1: Data Preparation', 'Phases of the Project > Phase 2: Model Training and Evaluation', 'Phases of the Project > Phase 3: Model Deployment Pipeline', 'Phases of the Project > Phase 4: Model Monitoring and Maintenance', 'Conclusion'], keywords=['Azure Machine Learning', 'model training', 'data preprocessing', 'feature engineering', 'model selection', 'model deployment', 'GitHub Actio

In [ ]:
tasks = set()

for dm in list_docminer:
    for t in dm.tasks:
        # tasks.add((dm.file_location, t.task_code, t.task_description))
        tasks.add((t.task_code, t.task_description))
        
tasks

{('DC0',
  'Recording and gathering video feeds from various remote desktop sessions.'),
 ('DP0',
  'Data Scientist, cleaning and preparing the video data for training purposes.'),
 ('DY0',
  'Integrating the trained model into a surveillance system for real-time monitoring.'),
 ('EV0',
  "Assessing the model's performance and fine-tuning it for optimal accuracy."),
 ('FE0',
  'Data Scientist, identifying and extracting relevant features from the video feeds that indicate normal and anomalous activities.'),
 ('MT0',
  'Data Scientist, utilizing machine learning algorithms to train the model on the labeled data.'),
 ('NA',
  'Implement monitoring tools to track model performance in production.')}

In [16]:
list_docminer_2 = []

# get list of files
doc_dir = "../linked_docs_v1"
for file in os.listdir(doc_dir):
    if file.endswith(".docx"):
        print(file)
        list_docminer_2.append(create_docminer(os.path.join(doc_dir, file), sync_openai))

list_docminer_2

Project Data Science Project for Remote Desktop Surveillance_v1.docx


[DocMiner(document_id='085818d7-ee61-4869-a5ab-dae782b03aab', document_filename='Project Data Science Project for Remote Desktop Surveillance_v1.docx', file_location='https://openaiembedding.blob.core.windows.net/docminer-container/Project Data Science Project for Remote Desktop Surveillance_v1.docx', project_title='Data Science Project for Remote Desktop Surveillance: Anomaly Detection Using AI', author=None, table_of_contents=['Purpose', 'Project Description', 'Intended Users', 'Model Development Environment', 'Potential Business Impact'], keywords=['remote desktop surveillance', 'anomaly detection', 'AI', 'video feeds', 'data collection', 'data preprocessing', 'feature extraction', 'model training', 'evaluation', 'validation', 'deployment', 'IT security', 'system administrators', 'data analysts', 'Python', 'TensorFlow', 'PyTorch', 'OpenCV', 'Scikit-learn', 'Jupyter Notebooks', 'AWS', 'Google Cloud', 'Git', 'data storage', 'cost reduction', 'increased efficiency', 'enhanced security'

In [21]:
tasks_2 = set()

for dm in list_docminer_2:
    for t in dm.tasks:
        # tasks_2.add((dm.file_location, t.task_code, t.task_description))
        tasks_2.add((t.task_code, t.task_description))
        
tasks_2

{('DC0',
  'Recording and gathering video feeds from various remote desktop sessions.'),
 ('DP0',
  'Collaborate with the enterprise data engineering team, work on a specification for cleaning and preparing the video data for training purposes. The specs for data pipelines and must manage by data engineering team.'),
 ('DY0',
  'Integrating the trained model into a surveillance system for real-time monitoring.'),
 ('EV0',
  "Assessing the model's performance and fine-tuning it for optimal accuracy."),
 ('FE0',
  'Data Scientist, identifying and extracting relevant features from the video feeds that indicate normal and anomalous activities.'),
 ('MT0',
  'Data Scientist, utilizing machine learning algorithms to train the model on the labeled data.')}

In [32]:
# compare tasks and tasks_2 by task_code and see if there is any difference of description
diff = tasks_2.difference(tasks)
diff

{('DP0',
  'Collaborate with the enterprise data engineering team, work on a specification for cleaning and preparing the video data for training purposes. The specs for data pipelines and must manage by data engineering team.')}

In [ ]:
# Find difference in task description
for d in diff:
    for original_task_id in tasks:
        if d[0] == original_task_id[0]:
            print(f"Original Task: {original_task_id[0]}: {original_task_id[1]}")
            print(f"New Task: {d[0]}: {d[1]}")
            print("\n")

Original Task: DP0: Data Scientist, cleaning and preparing the video data for training purposes.
New Task: DP0: Collaborate with the enterprise data engineering team, work on a specification for cleaning and preparing the video data for training purposes. The specs for data pipelines and must manage by data engineering team.




In [ ]:
# now we know difference in task description, we can update the task description in the document
# List of documents that need to be updated from list_docminer by task_code
